In [1]:
import chemfiles
import tensorflow as tf
import h5py

In [2]:
atoms = 5000

# Create a random Dataset

This random dataset contains about 1.4 GB or randomly generated data

In [ ]:
import numpy as np
import ase.io

ase.io.write("file.xyz",[ase.Atoms("H5000", positions=np.random.rand(atoms, 3)) for _ in range(5000)])

# Classic Approach
Read and write per configuration

In [3]:
def write_or_append(data, filename):
    try:
        with h5py.File(filename, "a") as hf:
            start_slice = hf["data"].shape[0]
            hf["data"].resize((start_slice + data.shape[0]), axis=0)
            hf["data"][start_slice:] = data
    except KeyError:
        with h5py.File(filename, "w") as f:
            f.create_dataset("data", data=data, maxshape=(None, atoms, 3),  chunks=(100, int(atoms / 10), 3))

In [4]:
%%time
with chemfiles.Trajectory('file.xyz') as traj:
    for frame in traj:
        write_or_append(frame.positions[None], filename="db.h5")

Wall time: 37.6 s


# Batch Approach

Read and write data in batches using a generator and tf.data API

In [5]:
def read_file():
    with chemfiles.Trajectory('file.xyz') as traj:
        for frame in traj:
            yield frame.positions

In [6]:
ds = tf.data.Dataset.from_generator(read_file, output_signature=tf.TensorSpec(shape=(atoms, 3), dtype=tf.float64))

In [7]:
%%time
for batch in ds.batch(16):
    write_or_append(batch, filename="tf_db.h5")

Wall time: 14.7 s


# Use Prefetching for even better performance

In [8]:
%%time
for batch in ds.batch(16).prefetch(tf.data.AUTOTUNE):
    write_or_append(batch, filename="tf_db_prefetch.h5")

Wall time: 12.5 s
